<a href="https://colab.research.google.com/github/patricnilackshan/UoM_DMS_Toolkit/blob/main/UoM_DMS_Toolkit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">

# 🎓 UoM DMS Toolkit

*Data-free downloads, uploads & sharing for UoM students*

**Torrents · YouTube · M3U8 · Direct Links**

---
*by [Patric Nilackshan](mailto:pnilackshan@gmail.com)*

</div>

---

## 🏫 Sign in to DMS

In [ ]:
%%capture
!pip install requests yt-dlp libtorrent
!apt-get -qq install -y p7zip-full
!curl -fsSL https://deno.land/install.sh | sh -s -- -y

import os
os.environ['PATH'] += os.pathsep + os.path.expanduser('~/.deno/bin')

#@markdown <center><img src='https://dms.uom.lk/apps/theming/image/logo' height="180"/></center>
#@markdown <center><h3>Sign in</h3></center><br>

UserName = "Enter your DMS UserName here..." # @param {type:"string"}
Password = "Enter your DMS Password here..." # @param {type:"string"}

LoginDetail = f'"{UserName}:{Password}"'
Header      = '"OCS-APIRequest: true"'
UploadPoint = '"https://dms.uom.lk/remote.php/webdav/"'
SharePoint  = '"https://dms.uom.lk/ocs/v2.php/apps/files_sharing/api/v1/shares?format=json"'

---

## 📥 Download File from URL

In [ ]:
#@markdown <center><img src='https://dms.uom.lk/svg/core/actions/public?color=000' height="120"/></center>
#@markdown <center><h3>Enter Download Link</h3></center><br>

Download_Link = "Enter your Download Link here..." # @param {type:"string"}

import os, sys, re, time, warnings, subprocess

FileName = "Download"

def download_torrent(link):
    import libtorrent as lt
    global FileName
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)
        ses = lt.session()
        handle = lt.add_magnet_uri(ses, link, {'save_path': './', 'storage_mode': lt.storage_mode_t(2)})
        sys.stdout.write('Fetching metadata...\n'); sys.stdout.flush()
        while not handle.has_metadata(): time.sleep(1)
        FileName = handle.get_torrent_info().name()
        sys.stdout.write(f'Downloading: {FileName}\nPress Ctrl+C to stop\n'); sys.stdout.flush()
        states = ['queued','checking','downloading metadata','downloading','finished','seeding','allocating','checking fastresume']
        while handle.status().state != lt.torrent_status.seeding:
            s = handle.status()
            sys.stdout.write('\r%.2f%% (↓ %.1f kB/s  ↑ %.1f kB/s  peers: %d)  %s' %
                (s.progress*100, s.download_rate/1000, s.upload_rate/1000, s.num_peers, states[s.state]))
            sys.stdout.flush(); time.sleep(1)
        files = [f.path for f in handle.get_torrent_info().files()]
        sys.stdout.write('\n✅ Download complete!\n\n'); sys.stdout.flush()
        if len(files) == 1:
            FileName = files[0]
            print(f'✅ {FileName} ready for upload')
        else:
            print('\n'.join(f"{i+1}: {f}" for i, f in enumerate(files)))
            while True:
                try:
                    sel = int(input('\nEnter 0 to ZIP all  OR  a file number to upload individually:\n'))
                    if sel == 0:
                        print('\n📦 Zipping...'); subprocess.run(['zip', '-r', f'{FileName}.zip', FileName], check=True)
                        FileName = f'{FileName}.zip'; print(f'✅ {FileName} ready for upload'); break
                    elif 1 <= sel <= len(files):
                        FileName = files[sel-1]; print(f'✅ {FileName} ready for upload'); break
                    else: print('⚠️ Invalid selection')
                except ValueError: print('⚠️ Please enter a number')

def download_file(link):
    import requests
    global FileName
    r = requests.get(link, stream=True)
    cd = r.headers.get('Content-Disposition', '')
    m = re.search(r'filename="?([^";\n]+)"?', cd)
    FileName = m.group(1).strip() if m else (os.path.basename(link.split('?')[0]) or 'Download')
    path = os.path.join(os.getcwd(), FileName)
    if r.status_code == 200:
        with open(path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=51200):
                if chunk: f.write(chunk)
        print(f'✅ Saved as {FileName}')
    else:
        print(f'⚠️ Failed. HTTP {r.status_code}')

def download_youtube(link):
    import yt_dlp
    global FileName
    ydl_opts = {'quiet': True, 'outtmpl': '%(title)s.%(ext)s', 'merge_output_format': 'mp4',
                'postprocessors': [{'key': 'FFmpegVideoConvertor', 'preferedformat': 'mp4'}]}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(link, download=False)
    formats = [f for f in info['formats'] if f.get('format_note') and f.get('ext')]
    print('\n'.join(f"{i+1}. {f['format_note']} - {f['ext']}" for i, f in enumerate(formats)))
    while True:
        try:
            c = int(input('\nSelect format number:\n')) - 1
            if 0 <= c < len(formats):
                ydl_opts['format'] = formats[c]['format_id'] + '+bestaudio'
                with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                    info_dict = ydl.extract_info(link, download=True)
                    FileName = os.path.splitext(ydl.prepare_filename(info_dict))[0] + '.mp4'
                print('\n✅ Video downloaded!'); break
            else: print('⚠️ Invalid selection')
        except ValueError: print('⚠️ Please enter a number')

def download_m3u8(link):
    global FileName
    FileName = 'stream.mp4'
    subprocess.run(['ffmpeg', '-threads', '8', '-i', link, '-c', 'copy', FileName], check=True)

link = Download_Link
if   link.startswith('magnet:'):                   download_torrent(link)
elif link.endswith('.m3u8'):                        download_m3u8(link)
elif 'youtube.com' in link or 'youtu.be' in link:  download_youtube(link)
else:                                               download_file(link)

print(f'\n📄 File ready: {FileName}')

---

## 📤 Upload File to DMS

In [ ]:
#@markdown <center><img src='https://dms.uom.lk/apps/circles/img/black_circle.svg' height="120"/></center>
#@markdown <center><h3>Upload to DMS</h3></center><br>

import re

bare = os.path.basename(FileName)
name, ext = os.path.splitext(bare)
SafeName = (re.sub(r'[^a-zA-Z0-9_.]', '_', name).strip('_') or 'file') + ext

src = FileName if os.path.isabs(FileName) else f'/content/{FileName}'
dst = f'/content/{SafeName}'

if bare != SafeName:
    os.rename(src, dst)
    print(f'Renamed: {bare} → {SafeName}')

FileName = dst

def upload_with_progress(cmd):
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)
    for line in proc.stderr:
        m = re.search(r'(\d+(?:\.\d+)?)%', line)
        if m:
            p = float(m.group(1))
            bar = '█' * int(50 * p / 100) + '░' * (50 - int(50 * p / 100))
            sys.stdout.write(f'\rUpload Progress: [{bar}] {p:.1f}%'); sys.stdout.flush()
    code = proc.stdout.read().strip()
    proc.wait()
    if code in ('200', '201', '204'):
        print('\n✅ Uploaded to DMS successfully.')
    elif code == '507':
        print('\n⚠️ Upload failed: Insufficient storage on DMS (HTTP 507).')
    else:
        print(f'\n⚠️ Upload failed (HTTP {code}).' if code else '\n⚠️ Upload failed.')

MAX_PART_SIZE = int(2.8 * 1024**3)  # DMS hard limit is 3GB; split above 2.8GB to stay safe

if os.path.getsize(FileName) > MAX_PART_SIZE:
    print(f'📦 File exceeds 2.8GB — splitting with 7-Zip...')
    result = subprocess.run(['7z', 'a', '-v2800m', f'{FileName}.7z', FileName])
    if result.returncode != 0:
        raise SystemExit(f'⚠️ 7-Zip split failed (exit code {result.returncode}).')
    parts = sorted(p for p in os.listdir(os.path.dirname(FileName)) if p.startswith(f'{SafeName}.7z.'))
    print(f'✅ Split into {len(parts)} parts.\n')
    for i, part in enumerate(parts, 1):
        input(f'Press Enter to upload part {i}/{len(parts)}: {part}...')
        part_path = os.path.join(os.path.dirname(FileName), part)
        upload_with_progress(f'curl -u {LoginDetail} --progress-bar -T "{part_path}" -o /dev/null --write-out "%{{http_code}}" "{UploadPoint}"')
else:
    upload_with_progress(f'curl -u {LoginDetail} --progress-bar -T "{FileName}" -o /dev/null --write-out "%{{http_code}}" "{UploadPoint}"')

---

## 🔗 Share File from DMS

In [ ]:
#@markdown <center><img src='https://dms.uom.lk/svg/core/actions/share?color=000' height="120"/></center>
#@markdown <center><h3>Share File</h3></center><br>

import json

body = f'"path={FileName[9:]}&shareType=3&permissions=1"'
resp = !curl -s -u {LoginDetail} -X POST -d {body} {SharePoint} -H {Header}
try:
    share = json.loads(''.join(resp))['ocs']['data']
    print(f"Share ID  : {share['id']}\nFile Path : {share['path']}\nDMS Link  : {share['url']}")
except:
    print('⚠️ Sharing failed.')

---

## 📋 All Shares in DMS Account

In [ ]:
#@markdown <center><img src='https://dms.uom.lk/svg/core/actions/tag?color=000' height="120"/></center>
#@markdown <center><h3>View All Shares</h3></center><br>

import json

resp = !curl -s -u {LoginDetail} -X GET {SharePoint} -H {Header}
try:
    shares = json.loads(''.join(resp))['ocs']['data']
    if shares:
        for s in shares:
            print(f"Share ID  : {s['id']}\nFile Path : {s['path']}\nDMS Link  : {s['url']}\n")
    else:
        print('No shares found.')
except:
    print('⚠️ Failed to retrieve shares.')

---

## 🗑️ Delete a Share

In [ ]:
#@markdown <center><img src='https://dms.uom.lk/core/img/actions/delete.svg' height="120"/></center>
#@markdown <center><h3>Delete Share</h3></center><br>

import json

Share_ID = 278342 # @param {type:"integer"}
endpoint = f'{SharePoint[:-13]}/{Share_ID}?format=json"'
resp = !curl -s -u {LoginDetail} -X DELETE {endpoint} -H {Header}
try:
    code = json.loads(''.join(resp))['ocs']['meta']['statuscode']
    print(f'✅ Share {Share_ID} deleted.' if code == 200 else '⚠️ Deletion failed.')
except:
    print('⚠️ Deletion failed.')

---

## 🖴 Mount DMS as a Local Drive

**Windows 🪟** — Run in Command Prompt:
```
net use Z: https://dms.uom.lk/remote.php/webdav/ /user:<userName> <userPassword>
```

**Linux 🐧** — Enter in File Manager → *Connect to Server*:
```
davs://dms.uom.lk/remote.php/webdav/
```

---

<div align="center">

© **Patric Nilackshan** · [pnilackshan@gmail.com](mailto:pnilackshan@gmail.com)

</div>